### **Student Information**
GitHub ID: FarRail

Kaggle name: Farrel G. M.

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Syntax:** `#` creates the largest heading (H1).

---
**Syntax:** `---` creates a horizontal rule (a separator line).

## 1. Model Development (10 pts Required)

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

I matched the tweet text in final_posts.json with the labels in emotion.csv using post_id. I used data_identification.csv to identify the training posts and sampleSubmission.csv to preserve the required test order. I converted the emotion labels into integer class IDs.

I created a stratified split with approximately 80% of the labeled data for training and 20% for validation, using seed 42. This kept examples of each emotion in both groups.

I kept the original tweet text, including punctuation and hashtags. For the fine-tuned MiniLM component, I tokenized the text, truncated sequences to 80 tokens, and padded each batch to its longest sequence.

For a separate clause-based component, I split posts at punctuation and words such as "but" and "however". For posts with many clauses, I retained the first three and last three clauses. This allowed the model to examine local expressions of emotion within longer posts.

### 1.2 Feature Engineering Steps

My final model used several types of features.

First, I generated normalized sentence embeddings with sentence-transformers/all-MiniLM-L6-v2. I also fine-tuned a separate MiniLM encoder and used attention-mask mean pooling to convert its token vectors into one vector per post.

Second, I created word unigram and bigram TF-IDF features, plus character n-grams of length 3 to 5. Both vectorizers used min_df=2 and sublinear term frequency. Their vocabularies and IDF weights were fitted on the training portion of each experiment.

Third, I created 22 binary features from patterns for emotion words, insults, threats, punctuation, uppercase words, the [NAME] placeholder, and first-person pronouns. I normalized these features before combining them with the sentence and TF-IDF similarities.

Finally, I generated normalized MiniLM embeddings for individual clauses. These provided a second view of the text that could retain emotional expressions within a longer post.

### 1.3 Explanation of Your Model

My final model (V12) used an ensemble approach by combining three different models to classify tweets into six emotion categories.

**1. Fine-tuned MiniLM**

I fine-tuned a pretrained MiniLM encoder with a linear classification layer to adapt its understanding of text to the competition dataset. The model used the AdamW optimizer, a batch size of 32, and six training epochs. The learning rate was set to 0.00003 for the encoder and 0.0005 for the classification layer, with both rates halved at epoch 5.

I used weighted cross-entropy loss to account for class imbalance and averaged the prediction probabilities from epochs 4 to 6 to reduce dependence on a single training epoch.

**2. Combined-feature SVM**

The second model used an SVM with a precomputed kernel that combined several types of similarity:

- MiniLM sentence embedding similarity (weight: 1.00)
- Emotion-related feature similarity (weight: 0.40)
- Word-level TF-IDF similarity (weight: 0.55)
- Character-level TF-IDF similarity (weight: 0.55)

The SVM used balanced class weights and a regularization parameter of C=1. Its prediction probabilities were combined equally with those of the fine-tuned MiniLM model to form the full-post predictions.

**3. Clause-level SVM**

The third model used an RBF SVM with C=3 to classify individual clauses within each tweet. The maximum and mean prediction probabilities across the clauses were combined to produce a prediction for the entire tweet.

The purpose of this approach was to capture specific expressions of emotion that might be overlooked when processing a tweet as a whole.

**4. Ensemble Weight Selection**

I tested four different weights for combining the full-post and clause-level predictions.

| Full-post weight | Clause-level weight | Validation macro F1 |
|---:|---:|---:|
| 100% | 0% | 0.5585 |
| **85%** | **15%** | **0.5639** |
| 70% | 30% | 0.5592 |
| 55% | 45% | 0.5586 |

The clause-level weight of 15% was selected because it achieved the highest validation macro F1 of 0.5639. A five-fold cross-fitted evaluation of the weight selection process produced a macro F1 of 0.5539.

The final contributions of each model were:

| Model | Contribution |
|---|---:|
| Fine-tuned MiniLM | 42.5% |
| Combined-feature SVM | 42.5% |
| Clause-level SVM | 15.0% |

After selecting these weights, I trained the models on all labeled training data and generated the final test predictions.

**5. Experimental Results**

| Version | Public Kaggle score |
|---|---:|
| V1 | 0.5482 |
| V2 | 0.5529 |
| V3 | 0.5504 |
| V4 | 0.5928 |
| V5 | 0.6106 |
| V6 | 0.6060 |
| V7 | 0.6121 |
| V10 | 0.6112 |
| **V12** | **0.6123** |

V12 was selected as my final submission and uploaded under the filename lab2_submission_v9.csv.

- Final public score: **0.6123**
- Final private score: **0.6135**
- Private leaderboard rank: **75**

From the results, I observed that additional training and changes to the model did not always guarantee better performance. Although the score improved considerably from V1 to V12, the later improvements became much smaller.

I also experimented with DeBERTa as an alternative encoder, which achieved a mean validation macro F1 of 0.5508 across three folds. I ultimately retained V12 because it achieved the highest recorded public and private leaderboard scores among my submissions.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used ChatGPT to help write model code, generate submission files, suggest improvements, and diagnose training errors. I supplied the competition files, reported submission scores, and shared validation output and error messages.

I used the suggested MiniLM fine-tuning and ensemble methods during development. The final v12 code combined fine-tuned MiniLM probabilities with SVM predictions based on sentence embeddings, TF-IDF features, emotion cues, and clause embeddings. I retained v12 as my selected submission because it had the highest recorded public score.

I also followed suggestions to test DeBERTa. This required fixing GPU setup and training problems, including predictions collapsing to one class and a non-finite loss. These experiments did not become my final selected submission.

The execution-related prompts are recorded below. Signup, initial file-access questions, navigation, upload instructions, and export requests are omitted. Long diagnostic output is shortened, while the requests and reported results are retained.

**Initial implementation and submission repair**

- “these are the data given, I have no idea how to do anything yet. Please help tell me what to do in the fastest and most effective way possible”
- “it says ID column id not found in submission”

**Model improvement requests and score feedback**

- “"Welcome to the leaderboard! Your submission scored 0.5482." is what they said, is there any way to get better?”
- “it gave a 0.5529, try to find a method to make it pass 0.6”
- “score given was 0.5504”
- “0.5928 was the new score, continue to train again”
- “it has changed to 0.6106, next benchmark is 0.64”
- “the new score is 0.6060, continue training to go pass the benbchmark”
- “the new score is now 0.6121”
- “continue to train until benchmark is passed”
- “this one got 0.6112, continue training”
- “this one's result is 0.6123, what needs to be done to improve substantially?”
- “continue training, this is my last submission”
- “see what else can be adjusted and continue training”
- “how come all this training leads to worse scores?”

**Training continuation and monitoring**

I also sent the following short execution requests during the experiments.

- “please continue training”
- “please inform me when its done”
- “okay continue training”
- “please continue”
- “status”
- “sorry to bother, please continue until training is finished and let me know about the progress”
- “continue training” — repeated six times in the exported record

**DeBERTa setup and debugging**

- “how to do this” — referring to the suggestion to test a stronger encoder on fresh validation splits
- “how to get/do this?”
- “got this error” — followed by the GPU availability assertion error
- “i did step 2 and got: [three training and validation fold sizes] what now?”
- “i got these values: DeBERTa fold scores: [0.0516, 0.0581, 0.0573] Mean macro F1: 0.0557 what else is needed?”
- “got this: [actual class counts] Predicted: Counter({'anger': 1124})”
- “got RuntimeError: Non-finite loss at step 2: nan”
- “give me the full code to paste and which cell to change”
- “so change def train_fold and rerun?”
- “the 24 example test got back this result: [training reached 24/24 correct] what then?”
- “got this: [training losses, class counts, and fold 1 validation macro F1 of 0.5448]”
- “git me the entire new code. Is it all in the new cell?”
- “i have 7 minuts, will there be enough time?”
- “[fold scores of 0.5448, 0.5722, and 0.5355, with mean macro F1 of 0.5508]”
- “try to train a new one asap, just try sisnce the previous try wont go away”

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [2]:
### Add the code related to the preprocessing steps in cells inside this section

## 2. Feature Engineering Steps

In [ ]:
### Add the code related to the feature engineering steps in cells inside this section

## 3. Model Implementation Steps

In [ ]:
### Add the code related to the model implementation steps in cells inside this section